# 13과 - Cognee 지식 그래프를 활용한 에이전트 메모리


## 설정

이 노트북은 [**Cognee**](https://www.cognee.ai/) 지식 그래프와 **Microsoft Agent Framework**(MAF)를 사용하여 지속적인 메모리를 가진 지능형 **코딩 어시스턴트**를 구축하는 방법을 보여줍니다.

Cognee는 비정형 텍스트를 벡터 임베딩으로 지원되는 구조화된 질의 가능한 지식 그래프로 변환하여 에이전트에 풍부하고 관계 인지적인 장기 기억을 제공합니다.

### 배울 내용
1. **지식 그래프 구축**: 개발자 프로필과 모범 사례를 구조화된 질의 가능한 지식으로 변환합니다.
2. **Cognee와 MAF 통합**: `@tool` 함수를 사용하여 MAF 에이전트가 Cognee의 지식 그래프를 질의하도록 합니다.
3. **세션 인식 대화**: 동일 세션 내 여러 질문에 걸쳐 컨텍스트를 유지합니다.
4. **장기 메모리**: 세션 간 중요한 지식을 지속하고 새로운 대화에서 이를 검색합니다.

### 사전 준비 사항
- Python 3.9 이상
- 세션 관리를 위한 로컬 Redis 실행 (`docker run -d -p 6379:6379 redis`)
- LLM API 키(예: OpenAI) — `.env`에 `LLM_API_KEY` 설정
- `.env`에 `CACHING=true` (Cognee 세션에 필요)
- 배포된 챗 모델이 있는 Azure AI Foundry 프로젝트
- `.env`에 `AZURE_AI_PROJECT_ENDPOINT` 및 `AZURE_AI_MODEL_DEPLOYMENT_NAME` 설정
- Azure CLI 인증 완료 (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## 에이전트 메모리 유형

이 노트북은 메인 13강 노트북에서 다룬 동일한 세 가지 메모리 유형을 탐구하지만, 장기 메모리 백엔드로 Cognee를 사용합니다:

| Memory Type | Mechanism | Lifetime |
|---|---|---|
| **작업 메모리** | `agent.create_session()` (MAF) | 단일 대화 |
| **단기 메모리** | Cognee 세션 캐시 (Redis) | 단일 세션 |
| **장기 메모리** | Cognee 지식 그래프 + 벡터 | 영구적 |

### Cognee의 메모리 아키텍처
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## 코그니 스토리지 준비하기


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Part 1 — 지식 베이스 구축

우리는 코딩 어시스턴트를 위한 포괄적인 지식 베이스를 만들기 위해 세 가지 유형의 데이터를 수집합니다:

1. **개발자 프로필** — 개인 전문 지식 및 기술 배경  
2. **Python 모범 사례** — 실제 지침과 함께하는 Python의 철학  
3. **과거 대화 기록** — 개발자와 AI 어시스턴트 간의 과거 Q&A 세션


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## 지식 그래프 시각화

Cognee는 추출한 엔터티와 관계에 대한 대화형 HTML 시각화를 렌더링할 수 있습니다.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memify로 메모리 강화하기

`memify()`는 지식 그래프를 분석하고 지능적인 규칙을 생성합니다 — 패턴, 모범 사례 및 개념 간의 관계를 식별합니다.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Part 2 — Cognee 도구가 포함된 MAF 에이전트

이제 `@tool` 함수를 통해 Cognee의 지식 그래프를 쿼리할 수 있는 MAF 에이전트를 만듭니다. 이를 통해 에이전트는 세션을 통해 대화 맥락을 유지하면서 그래프 인지 의미 검색의 전체 기능을 활용할 수 있습니다.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## 세션을 이용한 작업 메모리

`AgentSession`( `agent.create_session()`을 통해 생성됨)은 세션 내에서 작업 메모리를 제공합니다. 에이전트는 이전 메시지를 참조하는 동시에 Cognee의 장기 지식 그래프를 질의할 수 있습니다.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## 새 세션 — 장기 기억 지속

새로운 세션을 시작하면 작업 메모리가 지워지지만, Cognee 지식 그래프는 여전히 사용 가능합니다. 에이전트는 완전히 새로운 대화에서도 동일한 장기 지식을 검색할 수 있습니다.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## 요약

이 노트북에서는 **MAF의 작업 메모리**(`agent.create_session()`)와 **Cognee의 장기 지식 그래프**를 결합한 코딩 어시스턴트를 구축했습니다.

### 배운 내용
1. **지식 그래프 구축**: Cognee는 비정형 텍스트를 수집하여 그래프와 벡터 메모리를 만듭니다.
2. **memify로 그래프 확장**: 기존 그래프 위에 파생된 사실과 더 풍부한 관계를 추가합니다.
3. **MAF + Cognee 통합**: `@tool` 함수는 MAF 에이전트가 Cognee의 그래프를 자연스럽게 질의할 수 있게 합니다.
4. **작업 메모리 + 장기 메모리**: `AgentSession`(`agent.create_session()`을 통해) 세션 컨텍스트를 제공하며 Cognee는 지속적인 지식을 제공합니다.
5. **NodeSets로 필터링된 검색**: 지식 그래프의 특정 하위 집합(예: 원칙만)을 대상으로 합니다.

### 주요 요점
- **Cognee**는 원시 텍스트를 구조화되고 관계를 인식하는 메모리로 바꿔 단순 벡터 저장소보다 강력합니다.
- **`@tool` 함수**는 MAF 에이전트와 외부 지식 시스템 간을 깔끔하게 연결합니다.
- **`AgentSession`**(`agent.create_session()`을 통해)은 대화별 컨텍스트를 장기 지식과 분리하여 유지합니다.
- 동일한 지식 그래프가 여러 세션과 에이전트에 사용됩니다.

### 실제 활용 사례
- **개발자 코파일럿**: 코드 리뷰, 사고 분석, 아키텍처 어시스턴트
- **고객 지원 코파일럿**: 제품 문서, FAQ, CRM 노트를 통한 지원 에이전트
- **내부 전문가 코파일럿**: 정책, 법률, 보안 가이드라인 기반 어시스턴트
- **통합 데이터 레이어**: 구조화 데이터와 비구조화 데이터를 하나의 쿼리 가능한 그래프로 결합

### 다음 단계
- Cognee에서 시간 인식 실험
- 도메인별 그래프 품질을 위한 OWL 온톨로지 정의
- 피드백 루프 추가로 검색 성능 향상
- 동일 Cognee 메모리 레이어를 공유하는 다중 에이전트 시스템 확장


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:  
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 위해 노력하고 있으나, 자동 번역에는 오류나 부정확성이 포함될 수 있음을 양지하시기 바랍니다. 원래 문서의 원문이 권위 있는 자료로 간주되어야 합니다. 중요한 정보에 대해서는 전문 번역가의 번역을 권장합니다. 본 번역 사용으로 인한 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
